In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import proplot as pplt
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'tick.minor':False,
    'savefig.dpi':300,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'legend.fontsize':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

In [ ]:
def load_equations(runname):
    '''
    Load per-seed equation CSVs for a run and return a dict mapping seed -> DataFrame.
    Adds a 'seed' column to each DataFrame.
    '''
    seedframes = {}
    for seed in SEEDS:
        csvpath = os.path.join(MODELSDIR,'sr',f'{runname}_{seed}_equations.csv')
        if not os.path.exists(csvpath):
            print(f'Missing: {csvpath}')
            continue
        df = pd.read_csv(csvpath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes

def union_pareto(seedframes):
    '''
    Build a table with one row per (seed, complexity) and columns: seed, complexity, loss, equation.
    '''
    rows = []
    for seed,df in seedframes.items():
        for _,row in df.iterrows():
            rows.append(dict(seed=seed,complexity=int(row['complexity']),
                             loss=float(row['loss']),equation=str(row['equation'])))
    return pd.DataFrame(rows).sort_values(['seed','complexity']).reset_index(drop=True)

def print_equation_table(runname):
    seedframes = load_equations(runname)
    if not seedframes:
        print(f'No equations found for {runname}')
        return
    table = union_pareto(seedframes)
    print(f'\n=== {runname} ===')
    print(table.to_string(index=False,max_colwidth=80))

In [ ]:
def plot_pareto_overlay(runname,figpath=None):
    '''
    Plot the Pareto frontier (loss vs. complexity) for all seeds on a single axis,
    one line per seed, with scatter points at each equation node.
    '''
    seedframes = load_equations(runname)
    if not seedframes:
        print(f'No equations found for {runname}')
        return
    colors = ['blue9','red6','green7','yellow3','purple6']
    fig,ax = pplt.subplots(refwidth=5,refheight=2.5)
    ax.format(title=f'{runname}: Pareto Frontiers by Seed',
              xlabel='Equation Complexity (# nodes)',ylabel='Training Loss',grid=True)
    for i,(seed,df) in enumerate(sorted(seedframes.items())):
        df = df.sort_values('complexity')
        color = colors[i % len(colors)]
        ax.plot(df['complexity'],df['loss'],color=color,linewidth=1,linestyle='--',zorder=1)
        ax.scatter(df['complexity'],df['loss'],color=color,zorder=3,s=30,label=f'seed {seed}')
        best_idx = df['loss'].idxmin()
        ax.scatter([df.loc[best_idx,'complexity']],[df.loc[best_idx,'loss']],
                   color=color,zorder=4,s=80,marker='*')
    ax.legend(loc='ur',ncols=1)
    pplt.show()
    if figpath:
        fig.save(figpath)

In [ ]:
def plot_equation_table(runname,seed=None,figpath=None):
    '''
    Display a formatted table of equations for one seed alongside the Pareto frontier plot.
    '''
    seed       = seed if seed is not None else SEEDS[0]
    seedframes = load_equations(runname)
    if seed not in seedframes:
        print(f'No equations for {runname} seed {seed}')
        return
    df = seedframes[seed].sort_values('complexity').reset_index(drop=True)
    best_idx = int(df['loss'].idxmin())

    fig,axs = pplt.subplots(nrows=1,ncols=2,refwidth=3,refheight=max(2,0.35*len(df)+0.5),
                             wratios=(1,1.8),share=False)
    axs[0].format(title=f'Pareto Frontier (seed {seed})',
                  xlabel='Complexity',ylabel='Training Loss',grid=True)
    axs[0].plot(df['complexity'],df['loss'],color='gray5',linewidth=0.8,linestyle='--',zorder=1)
    axs[0].scatter(df['complexity'],df['loss'],color='green7',zorder=3,s=30)
    axs[0].scatter([df.loc[best_idx,'complexity']],[df.loc[best_idx,'loss']],
                   color='red6',zorder=4,s=80,marker='*',label='Best')
    axs[0].legend(loc='ur')

    axs[1].axis('off')
    colwidths = [0.15,0.15,0.70]
    headers   = ['Complexity','Loss','Equation']
    tabledata = [[str(int(row['complexity'])),f'{row["loss"]:.4f}',
                  (str(row['equation'])[:60]+'…') if len(str(row['equation']))>60 else str(row['equation'])]
                 for _,row in df.iterrows()]
    table = axs[1].table(cellText=tabledata,colLabels=headers,colWidths=colwidths,
                          loc='center',cellLoc='left')
    table.auto_set_font_size(False)
    table.set_fontsize(6)
    for (r,c),cell in table.get_celld().items():
        cell.set_edgecolor('gray8')
        if r==0:
            cell.set_facecolor('gray2')
            cell.set_text_props(color='white',fontweight='bold')
        elif r-1==best_idx:
            cell.set_facecolor('red1')
    axs[1].format(title=f'Equations (seed {seed})')
    fig.suptitle(runname)
    pplt.show()
    if figpath:
        fig.save(figpath)

In [ ]:
for runname in SRCONFIG['runs']:
    print_equation_table(runname)

In [ ]:
for runname in SRCONFIG['runs']:
    plot_pareto_overlay(runname)

In [ ]:
for runname in SRCONFIG['runs']:
    for seed in SEEDS:
        plot_equation_table(runname,seed=seed)